# ORBIT Gate 6 — Actor-Critic Policy Learning with World-Model Imagination

**Specification Reference:** `ORBIT_IMPLEMENTATION_BRIEF_v1.3.md` (Gate 6 Authorization)  
**Primary World Model:** Gate 4 Champion Formulation C (`results/gate4/formulation_c_residual_change-weighted.pt`)

## 1. Executive Summary & Gate 6 Objectives
In Gate 6, we train an actor-critic policy entirely within the imagination of our learned, frozen world model:
1. **Latent Imagination Rollouts:** Starting from real replay states ($z_0, h_0$), unroll the policy $\pi(a_t | z_t, h_t)$, predictor $\hat{z}_{t+1} = z_t + \Delta(z_t, a_t, h_t)$, and recurrent belief $h_{t+1} = \text{GRU}(z_{t+1}, a_t, r_t, h_t)$ across rollout horizons $H \in \{1, 2, 4, 8\}$.
2. **Generalized Advantage Estimation (GAE):** Compute advantages $\hat{A}_t^{\text{GAE}(\gamma=0.99, \lambda=0.95)}$ and train the value head via $\lambda$-return regression.
3. **Anti-Exploitation Safeguards:** Freeze all world model and encoder parameters strictly; clip predicted rewards to $[-1.0, 1.0]$; apply transition uncertainty penalties; compare imagined vs actual real-game returns.
4. **Real-Game Crafter Benchmarking:** Evaluate trained policies in unmodified Crafter across identical seeds ($4000..4009$) and benchmark against the matched Random Baseline.
5. **Strict Anti-Cheating Invariants:** Pixel-only agent boundary $(obs_{uint8}, reward_{prev}, step)$; zero access to privileged game state, inventory, or true achievements in decision loop.

## 2. Environment Setup & Dependency Installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Clone or update the repository
!git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world 2>/dev/null || (cd /content/learn-a-world && git pull origin main)
%cd /content/learn-a-world

# Install pinned dependencies
!pip install -q -r requirements.txt
!pip install -q torchvision timm matplotlib

## 3. Hardware Verification & GPU Acceleration Setup

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('WARNING: Running on CPU. For fast training, enable GPU runtime via Runtime -> Change runtime type.')

## 4. Run Verification Test Suite (Gates 0 through 6)

In [ ]:
%cd /content/learn-a-world
!pytest tests/test_orbit_gate6_policy.py tests/test_orbit_gate4_world_model.py tests/test_orbit_gate5_temporal_memory.py -v

## 5. Execute Gate 6 Experiment: Imagination Policy Training & Real Crafter Evaluation

This trains actor-critic policy heads over horizons $H \in \{1, 2, 4, 8\}$, runs anti-exploitation audits, benchmarks against the matched Random Baseline on unmodified Crafter, and saves checkpoints and telemetry.

In [ ]:
%cd /content/learn-a-world

!python experiments/run_gate6_experiment.py \
  --device auto \
  --updates 500 \
  --batch-size 16 \
  --actor-lr 1e-4 \
  --value-lr 1e-3 \
  --entropy-weight 0.01 \
  --reward-clip 1.0 \
  --eval-seeds 4000,4001,4002,4003,4004,4005,4006,4007,4008,4009 \
  --max-eval-steps 500 \
  --output-dir results/gate6

## 6. Visualize Comparison Plots & Anti-Exploitation Diagnostics

In [ ]:
import json
from IPython.display import Image, display

# Display 4-panel comparison plots
plot_path = 'results/gate6/gate6_policy_plots.png'
display(Image(plot_path))

# Print formatted results summary
with open('results/gate6/gate6_experiment_metrics.json') as f:
    results = json.load(f)

print('=' * 95)
print(f"{'Policy / Horizon':<25} | {'Mean Return':<12} | {'Crafter Score':<15} | {'Mean Steps':<12} | {'Entropy':<10}")
print('-' * 95)
rand = results['random_baseline']
print(f"{'Random Baseline':<25} | {rand['mean_return']:<12.2f} | {rand['crafter_score']:<15.4f}% | {rand['mean_length']:<12.1f} | {rand['action_entropy']:<10.3f}")
for h in [1, 2, 4, 8]:
    p = results[f'policy_h{h}']['real_game_eval']
    print(f"{'ORBIT Policy (H=' + str(h) + ')':<25} | {p['mean_return']:<12.2f} | {p['crafter_score']:<15.4f}% | {p['mean_length']:<12.1f} | {p['action_entropy']:<10.3f}")
print('=' * 95)

## 7. Backup Checkpoints & Metrics to Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/learn-a-world/results/gate6
!cp -r results/gate6/* /content/drive/MyDrive/learn-a-world/results/gate6/
print('Successfully backed up Gate 6 checkpoints and artifacts to Google Drive.')